<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h1 style="text-align: center;"><b>Домашнее задание. Обучение языковой модели с помощью LSTM </b></h1>


В этом задании Вам предстоит обучить языковую модель с помощью рекуррентной нейронной сети. В отличие от семинарского занятия, Вам необходимо будет работать с отдельными словами, а не буквами.


Установим модуль ```datasets```, чтобы нам проще было работать с данными.

In [ ]:
!pip install datasets

Импорт необходимых библиотек

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.model_selection import train_test_split
import nltk

from collections import Counter
from typing import List

import seaborn
seaborn.set(palette='summer')

In [4]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Подготовка данных

Воспользуемся датасетом imdb. В нем хранятся отзывы о фильмах с сайта imdb. Загрузим данные с помощью функции ```load_dataset```

In [6]:
# Загрузим датасет
dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Препроцессинг данных и создание словаря (1 балл)

Далее вам необходмо самостоятельно произвести препроцессинг данных и получить словарь или же просто ```set``` строк. Что необходимо сделать:

1. Разделить отдельные тренировочные примеры на отдельные предложения с помощью функции ```sent_tokenize``` из бибилиотеки ```nltk```. Каждое отдельное предложение будет одним тренировочным примером.
2. Оставить только те предложения, в которых меньше ```word_threshold``` слов.
3. Посчитать частоту вхождения каждого слова в оставшихся предложениях. Для деления предлоения на отдельные слова удобно использовать функцию ```word_tokenize```.
4. Создать объект ```vocab``` класса ```set```, положить в него служебные токены '\<unk\>', '\<bos\>', '\<eos\>', '\<pad\>' и vocab_size самых частовстречающихся слов.   

In [7]:
dataset['train']['text'][0]

'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, ev

In [8]:
sentences = []
word_threshold = 32

for split in ['train', 'unsupervised']:
    for example in tqdm(dataset[split]):
        text = example['text'].replace('<br />', ' ')
        for s in sent_tokenize(text):
             s = s.lower()
             if len(s.split()) < word_threshold:
                 sentences.append(s)


In [9]:
sentences[:5]

['i rented i am curious-yellow from my video store because of all the controversy that surrounded it when it was first released in 1967.',
 'the plot is centered around a young swedish drama student named lena who wants to learn everything she can about life.',
 'in between asking politicians and ordinary denizens of stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.',
 'what kills me about i am curious-yellow is that 40 years ago, this was considered pornographic.',
 "really, the sex and nudity scenes are few and far between, even then it's not shot like some cheaply made porno."]

In [10]:
print("Всего предложений:", len(sentences))

Всего предложений: 811365


Посчитаем для каждого слова его встречаемость.

In [11]:
words = Counter()

for sentence in sentences:
    for word in sentence.split():
        words[word] += 1
# Расчет встречаемости слов

Добавим в словарь ```vocab_size``` самых встречающихся слов.

In [12]:
vocab = set()
vocab_size = 40000

vocab = [word for word, count in words.most_common(vocab_size)]
vocab += ['<unk>', '<bos>', '<eos>', '<pad>']

In [13]:
print("Всего слов в словаре:", len(vocab))

Всего слов в словаре: 40004


### Подготовка датасета (1 балл)

Далее, как и в семинарском занятии, подготовим датасеты и даталоадеры.

В классе ```WordDataset``` вам необходимо реализовать метод ```__getitem__```, который будет возвращать сэмпл данных по входному idx, то есть список целых чисел (индексов слов).

Внутри этого метода необходимо добавить служебные токены начала и конца последовательности, а также токенизировать соответствующее предложение с помощью ```word_tokenize``` и сопоставить ему индексы из ```word2ind```.

In [14]:
word2ind = {char: i for i, char in enumerate(vocab)}
ind2word = {i: char for char, i in word2ind.items()}

In [15]:
class WordDataset:
    def __init__(self, sentences):
        self.data = sentences
        self.unk_id = word2ind['<unk>']
        self.bos_id = word2ind['<bos>']
        self.eos_id = word2ind['<eos>']
        self.pad_id = word2ind['<pad>']

    def __getitem__(self, idx: int) -> List[int]:
        tokenized_sentence = [self.bos_id]
        tokenized_sentence.extend(word2ind.get(x, self.unk_id) for x in self.data[idx].split())
        tokenized_sentence += [self.eos_id]
        

        return tokenized_sentence

    def __len__(self) -> int:
        return len(self.data)

In [16]:
def collate_fn_with_padding(
    input_batch: List[List[int]], pad_id=word2ind['<pad>']) -> torch.Tensor:
    seq_lens = [len(x) for x in input_batch]
    max_seq_len = max(seq_lens)

    new_batch = []
    for sequence in input_batch:
        for _ in range(max_seq_len - len(sequence)):
            sequence.append(pad_id)
        new_batch.append(sequence)

    sequences = torch.LongTensor(new_batch).to(device)

    new_batch = {
        'input_ids': sequences[:,:-1],
        'target_ids': sequences[:,1:]
    }

    return new_batch

In [17]:
train_sentences, temp_sentences = train_test_split(sentences, test_size=0.2, random_state=42)
eval_sentences, test_sentences = train_test_split(temp_sentences, test_size=0.5, random_state=42)

train_dataset = WordDataset(train_sentences)
eval_dataset = WordDataset(eval_sentences)
test_dataset = WordDataset(test_sentences)

batch_size = 128

train_dataloader = DataLoader(
    train_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

eval_dataloader = DataLoader(
    eval_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

test_dataloader = DataLoader(
    test_dataset, collate_fn=collate_fn_with_padding, batch_size=batch_size)

## Обучение и архитектура модели

Вам необходимо на практике проверить, что влияет на качество языковых моделей. В этом задании нужно провести серию экспериментов с различными вариантами языковых моделей и сравнить различия в конечной перплексии на тестовом множестве.

Возмоэные идеи для экспериментов:

* Различные RNN-блоки, например, LSTM или GRU. Также можно добавить сразу несколько RNN блоков друг над другом с помощью аргумента num_layers. Вам поможет официальная документация [здесь](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
* Различные размеры скрытого состояния. Различное количество линейных слоев после RNN-блока. Различные функции активации.
* Добавление нормализаций в виде Dropout, BatchNorm или LayerNorm
* Различные аргументы для оптимизации, например, подбор оптимального learning rate или тип алгоритма оптимизации SGD, Adam, RMSProp и другие
* Любые другие идеи и подходы

После проведения экспериментов необходимо составить таблицу результатов, в которой описан каждый эксперимент и посчитана перплексия на тестовом множестве.

Учтите, что эксперименты, которые различаются, например, только размером скрытого состояния или количеством линейных слоев считаются, как один эксперимент.

Успехов!

### Функция evaluate (1 балл)

Заполните функцию ```evaluate```

In [18]:
def evaluate(model, criterion, dataloader) -> float:
    model.eval()
    perplexity = []
    with torch.no_grad():
        for batch in dataloader:
            logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
            loss = criterion(logits, batch['target_ids'].flatten())
            perplexity.append(torch.exp(loss).item())

    perplexity = sum(perplexity) / len(perplexity)

    return perplexity

### Train loop (1 балл)

Напишите функцию для обучения модели.

In [19]:
def train_model(model, criterion, optimizer, num_epochs=5):
    losses = []
    perplexities = []
    
    for epoch in range(num_epochs):
        epoch_losses = []
        model.train()
        for batch in tqdm(train_dataloader):
            optimizer.zero_grad()
            logits = model(batch['input_ids']).flatten(start_dim=0, end_dim=1)
            loss = criterion(logits, batch['target_ids'].flatten(start_dim=0, end_dim=1))
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())
    
        losses.append(sum(epoch_losses) / len(epoch_losses))
        perplexities.append(evaluate(model, criterion, eval_dataloader))
        print(f"epoch {epoch+1}: loss={losses[-1]:.3f}, ppl={perplexities[-1]:.2f}")
    
    return losses, perplexities
    

### Первый эксперимент (2 балла)

Определите архитектуру модели и обучите её.

In [19]:
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        x = self.embedding(input_batch)
        x, _ = self.lstm(x)
        logits = self.linear(x)
        return logits

In [20]:
model = LanguageModel(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model.parameters())

In [21]:
train_model(model, criterion, optimizer)

epoch 1: loss=5.419, ppl=155.64


epoch 2: loss=4.876, ppl=131.88


epoch 3: loss=4.693, ppl=125.41


epoch 4: loss=4.581, ppl=123.71


epoch 5: loss=4.498, ppl=124.07


([5.41886365159829,
  4.875559691186959,
  4.6934459067011485,
  4.580781962640278,
  4.497610691829059],
 [155.63940008506412,
  131.88357969939898,
  125.41170323609553,
  123.70774432161252,
  124.07082153946067])

In [22]:
test = evaluate(model, criterion, test_dataloader)
print('test:', test)

test: 125.0306882407011


### Второй эксперимент (2 балла)

Попробуйте что-то поменять в модели или в пайплайне обучения, идеи для экспериментов можно подсмотреть выше.

In [20]:
class LanguageModelGRU(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        x = self.embedding(input_batch)
        x, _ = self.gru(x)
        logits = self.linear(x)
        return logits

In [21]:
model = LanguageModelGRU(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model.parameters())

In [22]:
train_model(model, criterion, optimizer)

epoch 1: loss=5.357, ppl=150.12


epoch 2: loss=4.845, ppl=133.29


epoch 3: loss=4.675, ppl=129.77


epoch 4: loss=4.569, ppl=130.01


epoch 5: loss=4.492, ppl=131.82


([5.35655710420774,
  4.845192525947132,
  4.675389689507921,
  4.569193787614254,
  4.491904725693754],
 [150.11556021272196,
  133.2939203785797,
  129.77434293103144,
  130.01353898635045,
  131.8200292497027])

In [23]:
test = evaluate(model, criterion, test_dataloader)
print('test:', test)

test: 132.47574712951854


In [24]:
class LanguageModelGRU(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True, num_layers=2)
        self.linear = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_batch: torch.Tensor) -> torch.Tensor:
        x = self.embedding(input_batch)
        x, _ = self.gru(x)
        logits = self.linear(x)
        return logits

In [25]:
model = LanguageModelGRU(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=word2ind['<pad>'])
optimizer = torch.optim.Adam(model.parameters())

In [26]:
train_model(model, criterion, optimizer)

epoch 1: loss=5.639, ppl=160.77


epoch 2: loss=4.905, ppl=133.24


epoch 3: loss=4.708, ppl=128.50


epoch 4: loss=4.594, ppl=127.89


epoch 5: loss=4.513, ppl=129.65


([5.638847776673946,
  4.90498838806378,
  4.708315692536463,
  4.593943496372797,
  4.512677143518857],
 [160.76513787398954,
  133.23906530867615,
  128.5049994676271,
  127.89280742801702,
  129.6547400658063])

In [27]:
test = evaluate(model, criterion, test_dataloader)
print('test:', test)

test: 130.0401444540415


### Отчет (2 балла)

Опишите проведенные эксперименты. Сравните перплексии полученных моделей. Предложите идеи по улучшению качества моделей.

### Эксперименты
```
№	Модель	        Perplexity 
1	LSTM        	125.03
2   GRU (1 слой)  	132.48
3   GRU (2 слоя)        130.04
```

Лучший результат показала однослойная LSTM - на 7 пунктов лучше однослойной GRU. Добавление второго слоя существенно не помогло (прирост на 2 пункта по сравнению с однослойной GRU). 

Модели переобучаются. Лосс падает на протяжении 5 эпох, а Перплексия перестает снижаться после 3-4 эпохи.

Идеи для улучшения:
1. Добавить регуляризацию (dropout-слой)
2. Добавить Gradient Clipping
3. Увеличить размерность скрытого слоя
